# 03 Feature Engineering

Objective: Create technical indicators (RSI, SMA, MACD) and prepare the feature matrix.

In [3]:
import pandas as pd
import numpy as np

In [58]:
# --- STEP 1: Re-load the clean, contiguous data ---
df = pd.read_csv(r'C:\Users\David\BTC-trend-prediction\data\raw\coin_Bitcoin.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df.drop(columns=['SNo','Name','Symbol'], inplace=True)
# --- STEP 2: Engineer Features (The Math) ---


In [59]:
# 1. Log Returns & Volatility (Stationary metrics)
df['Log_Ret'] = np.log(df['Close'] / df['Close'].shift(1))
df['Vol_20'] = df['Log_Ret'].rolling(window=20).std()


# 2. Relative Moving Averages (Distances)
df['MA13_Dist'] = (df['Close'] - df['Close'].rolling(13).mean()) / df['Close'].rolling(13).mean()
df['MA21_Dist'] = (df['Close'] - df['Close'].rolling(21).mean()) / df['Close'].rolling(21).mean()

# 3. Momentum (Z-Score & RSI)
df['Z_Score'] = (df['Close'] - df['Close'].rolling(20).mean()) / (df['Close'].rolling(20).std())

delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=5).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=5).mean()
df['RSI_5'] = 100 - (100 / (1 + (gain / loss)))

In [63]:
# --- STEP 3: Whale Detection (VSA Features) ---
# RVOL: Suspicious Volume
df['RVOL'] = df['Volume'] / df['Volume'].rolling(window=20).mean()
# VP_Divergence: Is volume backing the move?
df['VP_Divergence'] = df['Log_Ret'] * df['RVOL']
# OBV_Slope: Accumulation/Distribution
df['OBV'] = (np.sign(df['Log_Ret']) * df['Volume']).cumsum()
df['OBV_Slope'] = df['OBV'].diff(5) / df['OBV'].rolling(20).std()
# Absorption: Large volume on small price spread
df['Vol_Efficiency'] = (df['High'] - df['Low']) / df['Volume']

In [64]:
# --- STEP 3: Define the Target (3-Day Horizon) ---
# We want the model to predict if the price will be > 1.5% higher in 3 days
df['Future_3D_Ret'] = (df['Close'].shift(-3) - df['Close']) / df['Close']
df['Target'] = (df['Future_3D_Ret'] > 0.015).astype(int)

# --- STEP 4: Cleanup & The "Price Blind" Purge ---
# Drop all NaNs from rolling windows and shifts
df = df.dropna()

In [65]:
# --- STEP 5: Final Clean & Save ---
df = df.dropna()

features_v4 = [
    'Date', 'Target', 'Log_Ret', 'Vol_20', 'MA13_Dist', 'MA21_Dist', 
    'Z_Score', 'RSI_5', 'RVOL', 'VP_Divergence', 'OBV_Slope', 'Vol_Efficiency'
]

df_final = df[features_v4]
save_path = r'C:\Users\David\BTC-trend-prediction\data\processed\btc_alpha_v4.csv'
df_final.to_csv(save_path, index=False)

In [32]:
df.tail()

,Date,Close,Daily_Return,Daily_Range,SMA_21,SMA_13,RSI_5,Next_Day_Return,Target,SMA_13_Dist,SMA_21_Dist
2982,2021-06-28 23:59:59,34434.335314,-0.006214,0.038270,35671.795761,34536.756458,54.945647,0.041628,1,-0.002966,-0.034690
2983,2021-06-29 23:59:59,35867.777735,0.041628,0.063835,35785.850332,34346.042189,57.841894,-0.023055,0,0.044306,0.002289
2984,2021-06-30 23:59:59,35040.837249,-0.023055,0.056751,35676.122511,34114.298580,81.007034,-0.041915,0,0.027160,-0.017807
2985,2021-07-01 23:59:59,33572.117653,-0.041915,0.064107,35527.051953,33943.904185,60.813732,0.009679,1,-0.010953,-0.055027
2986,2021-07-02 23:59:59,33897.048590,0.009679,0.034484,35363.368575,33811.687210,41.186044,0.022760,1,0.002525,-0.041464
